# Lab 05: Predict Late Deliveries

**AIGC 5005 — QuickBite**

**Objective:** Predict late deliveries when orders are placed and reduce the financial cost of voucher decisions.

A flagged order costs $4. An unflagged late order costs $15.

The project uses a stratified 75/25 train/test split, leak-free preprocessing, five-fold cross-validation, model comparison, tuning, and a business-cost-based threshold.

The final test set is reserved for evaluation. This notebook reads the saved final test results rather than re-evaluating or tuning on the test set.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import ConfusionMatrixDisplay

from latedeliveries.data import make_orders

root = Path.cwd()
if not (root / "reports" / "metrics.json").exists():
    root = root.parent

orders = make_orders()
cv = pd.read_csv(root / "reports" / "cv_results.csv")
costs = pd.read_csv(root / "reports" / "threshold_costs.csv")
candidate_costs = pd.read_csv(root / "reports" / "model_cost_comparison.csv")
metrics = json.loads((root / "reports" / "metrics.json").read_text())
info = json.loads((root / "models" / "model_info.json").read_text())

print("Orders:", len(orders))
print("Late orders:", int(orders["late"].sum()))
print("Late rate:", f"{orders['late'].mean():.1%}")
display(orders.isna().sum().to_frame("Missing values"))

In [ ]:
display(cv.round(4))
display(candidate_costs)

print("Selected model:", info["model_name"])
print("Best hyperparameters:", info["best_settings"])
print("Selected threshold:", info["threshold"])

# Generate the complete CV table for the README.
names = ["accuracy", "precision", "recall", "f1", "roc_auc"]
header = "| Model | " + " | ".join(names) + " |"
separator = "|---" * 6 + "|"

rows = [header, separator]
for _, row in cv.iterrows():
    entries = [f"{row[f'{m}_mean']:.3f} ± {row[f'{m}_std']:.3f}" for m in names]
    rows.append("| " + " | ".join([row["model"], *entries]) + " |")

print("\nCOPY THIS TABLE INTO README:")
print("\n".join(rows))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(
    costs["threshold"],
    costs["cost"],
    marker="o",
)
ax.axvline(
    metrics["threshold"],
    linestyle="--",
    label=f"Chosen threshold = {metrics['threshold']:.2f}",
)

ax.set_title("Business Cost by Decision Threshold")
ax.set_xlabel("Probability Threshold")
ax.set_ylabel("Total Training CV Cost (USD)")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
cm = np.asarray(metrics["confusion_matrix"])

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["On time", "Late"],
).plot(values_format="d", colorbar=False)

plt.title("Final Held-Out Test Confusion Matrix")
plt.tight_layout()
plt.show()

print("Precision:", round(metrics["precision"], 3))
print("Recall:", round(metrics["recall"], 3))
print("F1:", round(metrics["f1"], 3))
print("ROC-AUC:", round(metrics["roc_auc"], 3))

In [ ]:
policy_costs = metrics["costs_usd"]

labels = ["Flag Nobody", "Flag Everybody", "Model"]
values = [
    policy_costs["flag_nobody"],
    policy_costs["flag_everybody"],
    policy_costs["model"],
]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, values)

ax.bar_label(
    bars,
    labels=[f"${v:,.0f}" for v in values],
    padding=3,
)
ax.set_ylim(0, max(values) * 1.18)
ax.set_ylabel("Total Cost (USD)")
ax.set_title("Final Test: Cost of Three Policies")

plt.tight_layout()
plt.show()

print(
    "Savings versus flag nobody:",
    f"${metrics['savings_vs_nobody_usd']:,.0f}",
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from latedeliveries.pipeline import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    RANDOM_STATE,
    build_pipeline,
    split_orders,
)

X_train, _, y_train, _ = split_orders(orders)
folds = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

safe_model = build_pipeline(LogisticRegression(max_iter=1000))
safe_auc = cross_val_score(
    safe_model,
    X_train,
    y_train,
    cv=folds,
    scoring="roc_auc",
).mean()

leaky_X = X_train.copy()
leaky_X["customer_rating"] = orders.loc[X_train.index, "customer_rating"]

num = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
prep = ColumnTransformer(
    [
        ("num", num, NUMERIC_FEATURES + ["customer_rating"]),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES,
        ),
    ]
)
leaky_model = Pipeline(
    [
        ("preprocessor", prep),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)
leaky_auc = cross_val_score(
    leaky_model,
    leaky_X,
    y_train,
    cv=folds,
    scoring="roc_auc",
).mean()

print(f"Without leakage ROC-AUC: {safe_auc:.4f}")
print(f"With leakage ROC-AUC: {leaky_auc:.4f}")
print(f"Difference: {leaky_auc - safe_auc:+.4f}")

In [ ]:
p = metrics["costs_usd"]
better = p["model"] < min(p["flag_nobody"], p["flag_everybody"])

decision = (
    "Pilot the model with monitoring."
    if better
    else "Do not deploy the model yet; the baseline policy is cheaper."
)

recommendation = f"""
### Recommendation

**Decision:** {decision}

The model's held-out test cost is ${p["model"]:,},
compared with ${p["flag_nobody"]:,} for flagging nobody
and ${p["flag_everybody"]:,} for flagging everybody.
Its ROC-AUC is {metrics["roc_auc"]:.3f}.

**Threshold:** {metrics["threshold"]:.2f}, selected using
out-of-fold training predictions to minimize the $4
voucher and $15 missed-delay costs, without using the test set.

**Excluded data:** Order ID is an identifier; late is the
target; preparation time and customer rating are unavailable
at order placement. Vehicle and courier experience were
also excluded because courier assignment timing is unverified.

**Limitation:** This is simulated data. Before real deployment,
we must validate the model on actual orders, confirm feature
availability, and monitor both costs and prediction quality.
"""

display(Markdown(recommendation))